# Qiskit Fundamentals — A From-Scratch Deep Dive

*Prepared as certification-prep material for Qiskit v2.x*

### Who this is for
You already know Python and have played with quantum computing ideas before, but you want the **basics to be rock solid** — not "I copy-pasted a Bell state circuit once," but "I understand *why* every line does what it does." This notebook is written for that goal. It over-explains on purpose.

### What this notebook covers
1. The physics you actually need (qubits, superposition, measurement) — just enough, no more
2. `QuantumCircuit` fundamentals — registers, gates, drawing
3. Single-qubit gates, their matrices, and what they do geometrically
4. The `Statevector` class — simulating exactly, with no randomness
5. Measurement, sampling, and shots — where randomness *does* enter
6. Multi-qubit systems and Qiskit's qubit-ordering convention (a classic gotcha)
7. Two-qubit gates and entanglement (Bell states, GHZ states)
8. Qiskit v2 **Primitives** (`Sampler`, `Estimator`) — the modern way to run circuits
9. Transpilation basics
10. Exam-style gotchas and a mini capstone exercise

### How the exercises work
Exercise cells contain a `# TODO` and raise `NotImplementedError` until you fill them in. Solutions live in the **last section** of the notebook so you're not tempted to peek immediately — try each exercise for a few minutes first. Run the check cell after each exercise; it will tell you if your answer is right, not just whether your code runs.

### Environment
This notebook was built and tested against:


In [ ]:
import qiskit
import qiskit_aer
print("qiskit     :", qiskit.__version__)
print("qiskit-aer :", qiskit_aer.__version__)
# If you're on the certification track, you want qiskit >= 1.0 (the v2.x exam
# is written against the "Primitives-first" API introduced in Qiskit 1.0,
# which is what this notebook uses throughout).

---
## 1. The physics you need (and no more)

### 1.1 The qubit

A classical bit is 0 or 1. A **qubit** is a unit vector in a 2-dimensional complex vector space, written in **bra-ket (Dirac) notation**:

$$|\psi\rangle = \alpha|0\rangle + \beta|1\rangle, \qquad \alpha,\beta \in \mathbb{C}, \qquad |\alpha|^2 + |\beta|^2 = 1$$

- $|0\rangle$ and $|1\rangle$ are the **computational basis states** — the quantum analogues of classical 0 and 1. As column vectors:
$$|0\rangle = \begin{pmatrix}1\\0\end{pmatrix}, \qquad |1\rangle = \begin{pmatrix}0\\1\end{pmatrix}$$
- $\alpha$ and $\beta$ are **amplitudes**. They are complex numbers, not probabilities.
- $|\alpha|^2$ is the *probability* of measuring the qubit as $0$; $|\beta|^2$ is the probability of measuring it as $1$. This is the **Born rule**, and the normalization condition $|\alpha|^2+|\beta|^2=1$ just says "probabilities sum to 1."
- When $\alpha$ and $\beta$ are both nonzero, the qubit is in **superposition** — not "0 and 1 at the same time" in any classical sense, but a genuinely different kind of object whose measurement statistics happen to look probabilistic.

### 1.2 Why amplitudes, not just probabilities?

Because amplitudes can be **negative or complex and they interfere**. Two paths to the same outcome can cancel (destructive interference) or reinforce (constructive interference) — this is the entire source of quantum algorithmic speedups. A probability distribution alone can never do this; that's the real reason qubits aren't "just probabilistic bits."

### 1.3 The Bloch sphere

Any single-qubit pure state can be visualized as a point on the surface of a unit sphere (the **Bloch sphere**):

$$|\psi\rangle = \cos\!\left(\frac{\theta}{2}\right)|0\rangle + e^{i\phi}\sin\!\left(\frac{\theta}{2}\right)|1\rangle$$

- $\theta$ (polar angle) controls the 0-vs-1 probability split.
- $\phi$ (azimuthal angle) is the **relative phase** — invisible if you only ever measure in the computational basis, but very visible once you apply more gates. Relative phase is the thing that trips people up the most when they're new to this, so keep an eye on it throughout this notebook.
- $|0\rangle$ is the north pole, $|1\rangle$ is the south pole.

### 1.4 Measurement is destructive and probabilistic

Measuring $|\psi\rangle = \alpha|0\rangle+\beta|1\rangle$ in the computational basis:
- returns `0` with probability $|\alpha|^2$, or `1` with probability $|\beta|^2$;
- **collapses** the state to whichever basis state was observed — the superposition is gone after that.

This is why a single run of a quantum circuit tells you almost nothing; you need to run it many times ("shots") and build up statistics. We'll see this concretely in Section 5.

We'll now stop talking about physics in the abstract and start building circuits, referring back to these ideas as they come up in code.

---
## 2. Your first circuit

A `QuantumCircuit` is Qiskit's central object: a list of instructions (gates, measurements) acting on qubits and bits. Let's build the simplest non-trivial one: put a qubit into superposition and measure it.

In [ ]:
from qiskit import QuantumCircuit

# QuantumCircuit(n_qubits, n_classical_bits)
# n_qubits: how many qubits the circuit acts on (all start in |0>)
# n_classical_bits: how many classical bits are available to store measurement results
qc = QuantumCircuit(1, 1)

# The Hadamard gate H maps |0> -> (|0>+|1>)/sqrt(2), an equal superposition.
qc.h(0)          # apply H to qubit 0

# measure(qubit_index, classical_bit_index): measure qubit 0, store result in bit 0
qc.measure(0, 0)

# .draw() renders the circuit. output="mpl" gives a nicer matplotlib rendering;
# output="text" (the default) gives an ASCII-art version that always works, no plotting backend needed.
qc.draw(output="mpl")

In [ ]:
# The plain-text version is handy when you just want to sanity check a circuit quickly
print(qc.draw(output="text"))

Notice the circuit has:
- 1 **quantum wire** (top) — carries the qubit through gates,
- 1 **classical wire** (bottom, double line) — carries the measured bit into classical storage.

This distinction (quantum wires vs classical wires) matters: gates act on quantum wires, but once you measure onto a classical wire, that value is an ordinary classical bit forever after — no more superposition.

### Exercise 1

Build a circuit on **one qubit** that:
1. Applies an `X` gate (the quantum NOT — flips $|0\rangle \leftrightarrow |1\rangle$),
2. Applies an `H` gate,
3. Measures the qubit into a classical bit.

Store it in a variable called `exercise_1_circuit`.

In [ ]:
def build_exercise_1_circuit():
    # TODO: build and return the circuit described above
    raise NotImplementedError("Replace this with your circuit")

exercise_1_circuit = build_exercise_1_circuit()
exercise_1_circuit.draw(output="mpl")

In [ ]:
# --- Check your answer ---
from qiskit.quantum_info import Statevector

def check_exercise_1(qc):
    ops = [instr.operation.name for instr in qc.data]
    assert ops == ["x", "h", "measure"], f"Expected gate order [x, h, measure], got {ops}"
    assert qc.num_qubits == 1 and qc.num_clbits == 1, "Circuit should have exactly 1 qubit and 1 classical bit"
    print("Exercise 1 passed! Gate order and register sizes are correct.")

check_exercise_1(exercise_1_circuit)

---
## 3. Quantum and classical registers, explicitly

`QuantumCircuit(1, 1)` implicitly creates one quantum register and one classical register. For anything beyond toy examples, it's worth naming registers explicitly — it makes circuits self-documenting and is exactly how you'll compose circuits from sub-circuits on the exam and in real projects.

In [ ]:
from qiskit import QuantumRegister, ClassicalRegister, QuantumCircuit

# Named registers - useful once circuits get complex or are composed together
qr = QuantumRegister(2, name="q")
cr = ClassicalRegister(2, name="c")
qc = QuantumCircuit(qr, cr)

qc.h(qr[0])          # index into the register directly
qc.cx(qr[0], qr[1])  # CNOT: flips qr[1] iff qr[0] is |1>
qc.measure(qr, cr)   # measure the WHOLE register at once (qr[0]->cr[0], qr[1]->cr[1])

qc.draw(output="mpl")

A circuit can also have **multiple** quantum or classical registers, which Qiskit will draw as separate labelled blocks. This is common when a circuit has, say, "data qubits" and "ancilla qubits" that you want to reason about separately.

In [ ]:
data = QuantumRegister(2, name="data")
ancilla = QuantumRegister(1, name="ancilla")
result = ClassicalRegister(2, name="result")

qc_multi = QuantumCircuit(data, ancilla, result)
qc_multi.h(data[0])
qc_multi.cx(data[0], ancilla[0])
qc_multi.cx(ancilla[0], data[1])
qc_multi.measure(data, result)
qc_multi.draw(output="mpl")

---
## 4. Single-qubit gates in depth

Every quantum gate is a **unitary matrix** ($U^\dagger U = I$) — this is what makes it reversible and probability-preserving. Qiskit's `Operator` class lets you inspect the actual matrix behind any gate, which is invaluable for building intuition (and for the exam, which does ask about matrix forms).

### 4.1 The Pauli gates: X, Y, Z

These are the quantum analogues of "flip" operations, one for each axis of the Bloch sphere.

In [ ]:
from qiskit.quantum_info import Operator
import numpy as np
np.set_printoptions(precision=3, suppress=True)

for gate_name in ["x", "y", "z"]:
    qc_gate = QuantumCircuit(1)
    getattr(qc_gate, gate_name)(0)
    matrix = Operator(qc_gate).data
    print(f"{gate_name.upper()} gate matrix:")
    print(matrix)
    print()

- **X** ($\sigma_x$): $|0\rangle \leftrightarrow |1\rangle$. The quantum NOT / bit-flip.
- **Z** ($\sigma_z$): leaves $|0\rangle$ alone, sends $|1\rangle \to -|1\rangle$. A **phase flip** — invisible on $|0\rangle$/$|1\rangle$ measurement, but very visible once you're in superposition (it rotates the Bloch sphere point around the Z axis by 180°).
- **Y** ($\sigma_y$): a combined bit-and-phase flip, $Y = iXZ$.

### 4.2 The Hadamard gate

$$H = \frac{1}{\sqrt{2}}\begin{pmatrix}1 & 1\\ 1 & -1\end{pmatrix}$$

$H$ is the standard way to create superposition: $H|0\rangle = \frac{|0\rangle+|1\rangle}{\sqrt2}$ (often written $|+\rangle$), and $H|1\rangle = \frac{|0\rangle-|1\rangle}{\sqrt2}$ (written $|-\rangle$). Applying $H$ twice returns you to where you started ($H^2 = I$) — a good sanity check to run for yourself below.

### 4.3 Phase gates: S and T

These only touch $|1\rangle$'s phase, leaving $|0\rangle$ untouched:

$$S = \begin{pmatrix}1&0\\0&i\end{pmatrix} \quad (\text{a } 90° \text{ Z-rotation}), \qquad T = \begin{pmatrix}1&0\\0&e^{i\pi/4}\end{pmatrix} \quad (\text{a } 45° \text{ Z-rotation})$$

$S = T^2$ and $S^2 = Z$. These "small phase rotation" gates matter a lot in practice because $T$ (with $H$ and CNOT) forms a **universal gate set** — meaning any quantum computation whatsoever can be built from just these three.

In [ ]:
qc_gate = QuantumCircuit(1)
qc_gate.s(0)
print("S matrix:\n", Operator(qc_gate).data)

qc_gate = QuantumCircuit(1)
qc_gate.t(0)
print("\nT matrix:\n", Operator(qc_gate).data)

### 4.4 Rotation gates: RX, RY, RZ

These are *parameterized*, continuous versions of the Pauli gates — rotations by an arbitrary angle $\theta$ about the X, Y, or Z axis of the Bloch sphere:

$$R_Z(\theta) = \begin{pmatrix}e^{-i\theta/2} & 0 \\ 0 & e^{i\theta/2}\end{pmatrix}$$

(similarly for $R_X, R_Y$, with the sine/cosine mixing the other pair of axes). Every single-qubit unitary can be written as a product of rotations (up to a global phase), so these three gates — plus a global phase — are "universal" for single-qubit operations.

In [ ]:
from qiskit.visualization import plot_bloch_multivector
from qiskit.quantum_info import Statevector
import matplotlib.pyplot as plt

# Watch a qubit rotate: start at |0>, apply increasing RY rotations, print the resulting
# amplitudes and probabilities at each step so you can see the smooth interpolation
# between |0> and |1> that a rotation gate (as opposed to a discrete flip) gives you.
angles = [0, np.pi/4, np.pi/2, 3*np.pi/4, np.pi]  # 0, 45, 90, 135, 180 degrees
print(f"{'theta (rad)':>12} | {'amplitudes':>28} | {'P(0)':>6} | {'P(1)':>6}")
print("-" * 62)
for theta in angles:
    qc_gate = QuantumCircuit(1)
    qc_gate.ry(theta, 0)
    sv = Statevector(qc_gate)
    probs = sv.probabilities()
    print(f"{theta:12.3f} | {str(sv.data.round(3)):>28} | {probs[0]:6.3f} | {probs[1]:6.3f}")

# A single Bloch-sphere snapshot at theta = pi/2 (an equal superposition, but reached
# via rotation about the Y axis rather than via H -- notice it lands on the same
# point as H|0> because H is (up to global phase) a rotation into the +X direction):
qc_gate = QuantumCircuit(1)
qc_gate.ry(np.pi/2, 0)
plot_bloch_multivector(Statevector(qc_gate))

### Exercise 2 — predict, then verify

Before running the check cell, **write down on paper** what state $|\psi\rangle$ you expect from this sequence, starting from $|0\rangle$:

1. Apply `H`
2. Apply `Z`
3. Apply `H` again

(Hint: think about what $Z$ does to the $|+\rangle$ state that $H|0\rangle$ produces, then what $H$ does to *that*.)

Then fill in the circuit below and check whether your prediction matches Qiskit's `Statevector`.

In [ ]:
def build_exercise_2_circuit():
    # TODO: build a 1-qubit circuit implementing H, then Z, then H (no measurement)
    raise NotImplementedError("Replace this with your circuit")

exercise_2_circuit = build_exercise_2_circuit()
final_state = Statevector(exercise_2_circuit)
print("Final statevector:", final_state.data.round(3))
exercise_2_circuit.draw(output="mpl")

In [ ]:
# --- Check your answer ---
def check_exercise_2(qc):
    expected = Statevector.from_label("1")  # H, Z, H on |0> should land you exactly on |1>
    actual = Statevector(qc)
    assert actual.equiv(expected), f"Expected state equivalent to |1>, got {actual.data.round(3)}"
    print("Exercise 2 passed! H-Z-H on |0> gives |1> (up to global phase) — this identity, HZH = X, is worth remembering.")

check_exercise_2(exercise_2_circuit)

---
## 5. Exact simulation with `Statevector`

`qiskit.quantum_info.Statevector` computes the **exact** final quantum state of a circuit by matrix multiplication — no randomness, no "shots." This is only possible on a simulator (a real quantum computer can't hand you the amplitudes directly; that would violate the no-cloning theorem and the nature of measurement). It's an essential debugging tool: when a circuit isn't behaving as expected, check the `Statevector` before you even think about running it on hardware or a shot-based simulator.

Important: a circuit passed to `Statevector` **must not contain measurements** — measurement is a random, classical operation, and `Statevector` only simulates the pure unitary evolution.

In [ ]:
# Build the (|00> + |11>)/sqrt(2) Bell state and inspect it exactly
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)

sv = Statevector(qc)
print("Statevector:", sv.data.round(3))
print("Probabilities:", sv.probabilities_dict())
sv.draw(output="latex")

`Statevector` also supports starting from a specific basis label and evolving it manually, which is handy for quick "what does this gate do to this state" checks without building a full circuit:

In [ ]:
from qiskit.quantum_info import Statevector
from qiskit.circuit.library import HGate

initial = Statevector.from_label("0")   # |0>
evolved = initial.evolve(HGate())        # apply H directly to the Statevector
print("H|0> =", evolved.data.round(3))

---
## 6. Measurement, shots, and where randomness enters

`Statevector` gives you the *exact* quantum state, but that's not what a real quantum computer gives you. A real device (or a shot-based simulator standing in for one) can only give you **measurement outcomes**, sampled according to the Born-rule probabilities — and you only learn the probabilities themselves by repeating the circuit many times ("shots") and looking at the resulting frequency distribution.

Qiskit v2.x's recommended tool for this is the **Sampler primitive** (more on primitives generally in Section 9). For now, the exact-simulation version, `StatevectorSampler`, is enough to illustrate the idea — it samples from the exact statevector distribution with no hardware noise.

In [ ]:
from qiskit.primitives import StatevectorSampler

qc = QuantumCircuit(1, 1)
qc.h(0)
qc.measure(0, 0)

sampler = StatevectorSampler()
# .run() takes a list of "primitive unified blocs" (PUBs); a bare circuit is fine for the simplest case.
# shots=1: run the circuit exactly once
job = sampler.run([qc], shots=1)
result = job.result()
counts_1_shot = result[0].data.c.get_counts()
print("Result of a SINGLE shot:", counts_1_shot)
print("-> With only 1 shot you cannot tell this apart from a biased coin, or from noise.")

In [ ]:
# Now with many shots, the 50/50 nature of H|0> becomes clear
job = sampler.run([qc], shots=4000)
result = job.result()
counts = result[0].data.c.get_counts()
print("Result of 4000 shots:", counts)

from qiskit.visualization import plot_histogram
plot_histogram(counts, title="H|0> measured 4000 times")

**Key takeaway for the exam and for real debugging:** a shot count that looks "wrong" (e.g. 48%/52% instead of a clean 50/50) is completely normal --- it's statistical noise from a finite number of shots, not necessarily a bug or hardware error. More shots narrows the statistical uncertainty; it does not remove hardware noise (that requires error mitigation/correction, a topic beyond "basics").

### Exercise 3

Predict the measurement distribution (as fractions, not exact counts) you'd expect from measuring the state produced by `RY(pi/2)` applied to `|0\rangle`, then verify it by sampling.

In [ ]:
def build_exercise_3_circuit():
    # TODO: build a 1-qubit, 1-classical-bit circuit that applies RY(pi/2) to qubit 0
    # and then measures it into classical bit 0.
    raise NotImplementedError("Replace this with your circuit")

exercise_3_circuit = build_exercise_3_circuit()
sampler = StatevectorSampler()
job = sampler.run([exercise_3_circuit], shots=4000)
counts = job.result()[0].data.c.get_counts()
print(counts)
plot_histogram(counts)

In [ ]:
# --- Check your answer ---
def check_exercise_3(qc):
    ops = [instr.operation.name for instr in qc.data]
    assert "ry" in ops and "measure" in ops, "Circuit must contain an RY gate and a measurement"
    ry_instr = [instr for instr in qc.data if instr.operation.name == "ry"][0]
    theta = float(ry_instr.operation.params[0])
    assert abs(theta - np.pi/2) < 1e-6, f"Expected RY angle pi/2, got {theta}"
    print("Exercise 3 passed! RY(pi/2)|0> should give ~50/50 (0)/(1) — same split as H, "
          "different point on the Bloch sphere's equator, but measurement in the Z basis can't tell them apart.")

check_exercise_3(exercise_3_circuit)

---
## 7. Multi-qubit systems and Qiskit's ordering convention

This section covers the single most common source of confusion for people learning Qiskit, including experienced programmers coming from textbooks that use the opposite convention. Read it carefully.

### 7.1 Tensor products

An $n$-qubit state lives in a $2^n$-dimensional space, built as the **tensor product** of the individual qubit spaces. For 2 qubits:

$$|q_1\rangle \otimes |q_0\rangle = |q_1 q_0\rangle$$

### 7.2 Qiskit uses little-endian ("qubit 0 is the rightmost / least-significant bit")

Most textbooks write $|q_0 q_1 \dots q_{n-1}\rangle$ with qubit 0 **leftmost**. **Qiskit does the opposite**: qubit 0 is the **rightmost** bit in the ket label, and the **least significant bit** in the integer/binary interpretation of a basis state. This matches how binary numbers are normally written (bit 0 = least significant), which is why Qiskit chose it, but it's the opposite of a lot of quantum computing textbooks (e.g. Nielsen & Chuang).

Let's make this concrete instead of just asserting it.

In [ ]:
qc = QuantumCircuit(2)
qc.x(0)  # flip ONLY qubit 0 (the "first" qubit you defined)

sv = Statevector(qc)
print("State after X on qubit 0 only:", sv.probabilities_dict())
# Qiskit reports this as the state "01", meaning: qubit index 1 = 0, qubit index 0 = 1
# reading the string LEFT TO RIGHT as (qubit n-1, ..., qubit 1, qubit 0).
# In other words: the RIGHTMOST character of the bitstring is qubit 0.

In [ ]:
# Let's verify with 3 qubits and a distinct gate on each, so there is no ambiguity:
qc = QuantumCircuit(3)
qc.x(0)   # flip qubit 0 only
sv = Statevector(qc)
print("X on qubit 0 only ->", sv.probabilities_dict())   # expect '001': qubit0=1, others=0

qc = QuantumCircuit(3)
qc.x(2)   # flip qubit 2 only
sv = Statevector(qc)
print("X on qubit 2 only ->", sv.probabilities_dict())   # expect '100': qubit2=1, others=0

So reading a Qiskit bitstring like `'100'` for 3 qubits: that's qubit 2 = 1, qubit 1 = 0, qubit 0 = 0 — the **rightmost character is qubit 0**.

**Why this matters in practice:** if you build a circuit meant to compute, say, a 3-bit binary number where qubit 0 is your "ones place," your classical-bit interpretation lines up naturally with Qiskit's convention. But if you're translating a circuit diagram from a textbook that draws qubit 0 at the *top* and lists states as $|q_0 q_1 q_2\rangle$, you need to either reverse the bitstring when interpreting results, or be careful about which convention you're using end-to-end. This single issue causes more "my circuit doesn't match the textbook" confusion than anything else in an introductory course — including, per the certification exam's own emphasis, being explicitly tested.

---
## 8. Two-qubit gates

### 8.1 CNOT (controlled-X)

The workhorse two-qubit gate. `qc.cx(control, target)`: if the control qubit is $|1\rangle$, apply $X$ to the target; otherwise, do nothing. Its matrix (in Qiskit's little-endian convention, control = qubit 0, target = qubit 1, so the ordering of basis states is $|q_1 q_0\rangle$ = 00, 01, 10, 11):

In [ ]:
qc_gate = QuantumCircuit(2)
qc_gate.cx(0, 1)  # control=qubit0, target=qubit1
print("CNOT (control=q0, target=q1) matrix:")
print(Operator(qc_gate).data.real)
# Reading this: basis order is |00>, |01>, |10>, |11> (q1 q0).
# |10> (q0=1) means control is 1 -> target flips -> becomes |11>. Check row/col 2 and 3 swap.

### 8.2 CZ (controlled-Z), and the general pattern

`CZ` applies a $Z$ instead of an $X$ to the target when the control is $|1\rangle$. More generally, Qiskit lets you make *any* single-qubit gate controlled via `.control()`, and lets you build multi-controlled versions too (e.g. Toffoli / CCX = doubly-controlled X).

### 8.3 SWAP

Exchanges the states of two qubits entirely: $|q_1 q_0\rangle \to |q_0 q_1\rangle$. Useful when a hardware topology doesn't allow a two-qubit gate directly between the qubits you need (more on this in Section 11, transpilation).

In [ ]:
qc_demo = QuantumCircuit(2)
qc_demo.x(0)         # qubit 0 = |1>, qubit 1 = |0>
qc_demo.swap(0, 1)   # now qubit 0 should be |0>, qubit 1 should be |1>
print(Statevector(qc_demo).probabilities_dict())
qc_demo.draw(output="mpl")

---
## 9. Entanglement: Bell states and GHZ states

### 9.1 What entanglement actually means

A multi-qubit state is **entangled** if it *cannot* be written as a tensor product of individual single-qubit states — i.e., you cannot describe qubit A's state and qubit B's state separately; the description only makes sense for the pair as a whole. This is a purely quantum phenomenon with no classical analogue, and it's the resource that underlies quantum teleportation, superdense coding, and much of the advantage in quantum algorithms.

### 9.2 The four Bell states

The canonical maximally-entangled 2-qubit states:

$$|\Phi^+\rangle = \frac{|00\rangle+|11\rangle}{\sqrt2}, \quad |\Phi^-\rangle = \frac{|00\rangle-|11\rangle}{\sqrt2}, \quad |\Psi^+\rangle = \frac{|01\rangle+|10\rangle}{\sqrt2}, \quad |\Psi^-\rangle = \frac{|01\rangle-|10\rangle}{\sqrt2}$$

The standard recipe for $|\Phi^+\rangle$: Hadamard on qubit 0, then CNOT(0 -> 1).

In [ ]:
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
sv = Statevector(qc)
print("Phi+ :", sv.probabilities_dict())
print("Amplitudes:", sv.data.round(3))

# Proof of entanglement: try to factor this state as (a|0>+b|1>) tensor (c|0>+d|1>).
# If it factored, the reduced density matrix of either qubit would be a pure state
# (purity = 1). Let's check the purity of qubit 0 alone:
from qiskit.quantum_info import partial_trace
reduced_q0 = partial_trace(sv, [1])  # trace out qubit 1, leaving qubit 0's reduced state
print("\nReduced density matrix of qubit 0 alone:")
print(reduced_q0.data.round(3))
print("Purity:", reduced_q0.purity().real.round(3), " (1.0 = pure/unentangled, 0.5 = maximally mixed/entangled)")

The reduced state of qubit 0 is the **maximally mixed state** $I/2$ — purity $0.5$, the lowest possible for a single qubit. This is the mathematical signature of entanglement: qubit 0 *on its own* looks like a a completely random classical bit, even though the two-qubit system as a whole is in a perfectly well-defined pure state. All the information is in the *correlation* between the qubits, not in either qubit individually.

### 9.3 GHZ states: entanglement across 3+ qubits

The Greenberger-Horne-Zeilinger (GHZ) state generalizes this idea to $n$ qubits:

$$|\text{GHZ}\rangle = \frac{|00\dots0\rangle + |11\dots1\rangle}{\sqrt2}$$

Built the same way, just fanning the entanglement out from one qubit to all the others via a chain of CNOTs.

In [ ]:
def ghz_circuit(n_qubits):
    qc = QuantumCircuit(n_qubits)
    qc.h(0)
    for i in range(n_qubits - 1):
        qc.cx(i, i + 1)   # chain: 0->1, 1->2, 2->3, ...
    return qc

qc_ghz = ghz_circuit(3)
print(Statevector(qc_ghz).probabilities_dict())
qc_ghz.draw(output="mpl")

### Exercise 4

Build the Bell state $|\Psi^-\rangle = \frac{|01\rangle - |10\rangle}{\sqrt2}$ (this particular one is called the **singlet state** and shows up throughout physics). 

Hint: start from the standard $|\Phi^+\rangle$ recipe (H then CNOT) and figure out which extra single-qubit gate(s), applied *before* the CNOT, will turn it into $|\Psi^-\rangle$. Think about what needs to change: (a) which computational basis states appear (00,11 -> 01,10) and (b) the relative sign (+ -> -).

In [ ]:
def build_exercise_4_circuit():
    # TODO: build a 2-qubit circuit (no measurement) producing (|01> - |10>)/sqrt(2)
    raise NotImplementedError("Replace this with your circuit")

exercise_4_circuit = build_exercise_4_circuit()
sv = Statevector(exercise_4_circuit)
print(sv.data.round(3))
exercise_4_circuit.draw(output="mpl")

In [ ]:
# --- Check your answer ---
def check_exercise_4(qc):
    target = Statevector.from_label("01").data - Statevector.from_label("10").data
    target = target / np.linalg.norm(target)
    actual = Statevector(qc)
    # equiv() checks equality up to a global phase, which is unobservable and therefore fine
    assert actual.equiv(Statevector(target)), f"Got {actual.data.round(3)}, expected (|01>-|10>)/sqrt(2) up to global phase"
    print("Exercise 4 passed! You've built the singlet state |Psi->.")

check_exercise_4(exercise_4_circuit)

---
## 10. Qiskit v2 Primitives: `Sampler` and `Estimator`

If you've seen older Qiskit tutorials (or the v1.x exam), you may have encountered `backend.run(circuit)`. **That pattern is deprecated in the current API.** The v2.x certification is built around **Primitives**, which is the current recommended entry point for actually executing circuits. It's worth understanding *why* this changed, not just memorizing the new syntax.

### 10.1 Why primitives instead of `backend.run`?

`backend.run()` just returns raw counts and leaves you to compute anything derived (expectation values, error mitigation, etc.) yourself. Primitives instead model the **two things people actually want from a quantum computer**:

- **`Sampler`** — give me the probability distribution over measurement outcomes (bitstrings). This is what you use when the *bitstrings themselves* are the answer (e.g. Grover's algorithm, sampling problems).
- **`Estimator`** — give me the expectation value of an observable (e.g. a Hamiltonian's energy). This is what you use in variational algorithms like VQE and QAOA, where the number you actually care about is $\langle\psi|H|\psi\rangle$, not the raw bitstrings.

This separation lets Qiskit (and the underlying hardware provider) optimize execution and apply error mitigation appropriately for the kind of answer you're asking for, instead of you reconstructing it by hand from counts every time.

### 10.2 `Sampler` (we already used this in Section 6)

We used `StatevectorSampler` above, which is Qiskit's own ideal (noiseless) reference simulator implementation of the Sampler interface. `qiskit_aer.primitives.SamplerV2` is the Aer-backed equivalent, letting you plug in noise models, custom simulation methods (statevector / matrix-product-state / stabilizer), etc. The *interface* — `.run([circuit], shots=...)` -> job -> `.result()` — stays the same regardless of which one is underneath; that consistency of interface across simulators and real hardware is the whole point of the primitives design.

In [ ]:
from qiskit_aer.primitives import SamplerV2 as AerSampler

qc = QuantumCircuit(2, 2)
qc.h(0)
qc.cx(0, 1)
qc.measure([0, 1], [0, 1])

aer_sampler = AerSampler()
job = aer_sampler.run([qc], shots=2000)
counts = job.result()[0].data.c.get_counts()
print(counts)
plot_histogram(counts, title="Bell state via Aer-backed SamplerV2")

### 10.3 `Estimator` and observables

`Estimator` needs two things per circuit: the circuit itself (*without* measurements — expectation values are computed differently from sampling) and an **observable**, expressed as a `SparsePauliOp` — a weighted sum of tensor products of Pauli operators.

In [ ]:
from qiskit.quantum_info import SparsePauliOp
from qiskit.primitives import StatevectorEstimator

qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)   # Bell state, no measurement -- Estimator computes expectation values directly

# Observable: Z on qubit 0 tensor Z on qubit 1, i.e. "ZZ".
# For the Bell state, qubit 0 and qubit 1 are always perfectly correlated (00 or 11),
# so Z0*Z1 should have expectation value exactly +1 (since Z|0>=+|0>, Z|1>=-|1>,
# and the products +1*+1 and (-1)*(-1) are both +1).
observable = SparsePauliOp("ZZ")

estimator = StatevectorEstimator()
# A "PUB" (primitive unified bloc) for Estimator is (circuit, observable[, parameter_values])
job = estimator.run([(qc, observable)])
result = job.result()
print("<ZZ> expectation value:", result[0].data.evs)

# Contrast: for a Bell state, <Z0> alone (just measuring qubit 0's Z) should be 0,
# because qubit 0 alone is maximally mixed (Section 9.2) -- 50/50 between +1 and -1.
observable_z0 = SparsePauliOp("IZ")  # identity on qubit1, Z on qubit0 (remember: little-endian, rightmost = qubit 0)
job = estimator.run([(qc, observable_z0)])
print("<Z0> expectation value:", job.result()[0].data.evs, " (expected ~0)")

This "individually random, jointly perfectly correlated" pattern — $\langle Z_0\rangle \approx 0$ but $\langle Z_0 Z_1\rangle = 1$ exactly — is, numerically, the same entanglement signature we saw via the reduced density matrix purity in Section 9. Seeing it show up twice from two completely different calculations is a good way to build confidence you actually understand what's going on rather than having memorized one code path.

---
## 11. Transpilation basics

A circuit you write (using `h`, `cx`, `ry`, etc.) is usually **not** directly executable on a given real backend. Real hardware supports only a small **basis gate set** (e.g. maybe just `{RZ, SX, X, CX}`) and has a fixed **connectivity graph** (not every qubit can interact directly with every other qubit). **Transpilation** is the process of rewriting your circuit into an equivalent one that respects those constraints.

In [ ]:
from qiskit import transpile
from qiskit_aer import AerSimulator

qc = QuantumCircuit(3)
qc.h(0)
qc.cx(0, 1)
qc.cx(1, 2)
qc.ry(0.7, 2)

backend = AerSimulator()
# optimization_level: 0 (no optimization, fastest to transpile) through 3 (most aggressive
# optimization, slowest to transpile but often fewer gates / less depth in the output)
transpiled = transpile(qc, backend=backend, optimization_level=3)

print("Original circuit -- gate counts:", qc.count_ops(), " depth:", qc.depth())
print("Transpiled circuit -- gate counts:", transpiled.count_ops(), " depth:", transpiled.depth())
transpiled.draw(output="mpl")

Two numbers worth caring about after transpiling:
- **Depth** — the number of sequential gate "layers"; roughly, how long the circuit takes to run and how much noise accumulates on real hardware.
- **Gate counts**, especially **two-qubit gate count** — two-qubit gates are typically far noisier than single-qubit gates on real devices, so this is often the single best proxy for "how much will this circuit suffer from hardware noise."

On a simulator like `AerSimulator` with no coupling-map restriction, transpilation mostly just optimizes gate counts. On real backends (or a simulator configured with a specific `coupling_map`), transpilation will also insert `SWAP` gates where needed to route two-qubit gates between qubits that aren't directly connected — which is exactly why the `SWAP` gate from Section 8.3 matters in practice, not just as a toy operation.

---
## 12. Gotchas worth memorizing

A condensed list of the things that most commonly trip people up — both on certification exams and in real debugging sessions:

1. **Little-endian ordering.** Qubit 0 is the rightmost bit in a Qiskit basis-state string, and the least significant bit numerically. (Section 7.)
2. **`Statevector` cannot contain measurements.** Measurement is stochastic; `Statevector` only simulates the deterministic unitary part. Build two versions of a circuit if you need both an exact-state check *and* a shot-based histogram.
3. **Global phase is unobservable.** Two states differing only by an overall factor $e^{i\gamma}$ (not a *relative* phase between amplitudes) give identical measurement statistics for anything you can actually measure. This is why `Statevector.equiv()` checks equality "up to global phase" rather than exact array equality — and why HZH = X *up to global phase* is still a completely valid and useful identity.
4. **`backend.run()` is legacy.** The current API (and the v2.x exam) centers on `Sampler` and `Estimator` primitives. If you see `backend.run(circuit).result().get_counts()` in an old tutorial, translate it mentally to `Sampler().run([circuit]).result()[0].data.<creg_name>.get_counts()`.
5. **`Estimator` circuits should generally have no measurements**; it computes expectation values directly from the (simulated or estimated) state, not from measurement counts.
6. **A single shot tells you almost nothing.** Any probabilistic claim about a circuit's behavior requires enough shots to be statistically meaningful — one anomalous run is not evidence of a bug.
7. **Two-qubit gate count/depth after transpilation, not the code you wrote, determines noise sensitivity on real hardware.** A circuit that looks small in your source code can transpile into something much larger once routed to a real device's connectivity.
8. **Register naming vs. qubit index.** `QuantumRegister(2, name="q")[0]` and plain integer indexing into a `QuantumCircuit` ultimately refer to the same physical qubit slots — named registers are for readability/composition, not a different addressing scheme.

---
## 13. Capstone exercise

Put everything together. Build a circuit on **3 qubits** that:

1. Prepares the 3-qubit **GHZ state** $\frac{|000\rangle + |111\rangle}{\sqrt2}$ (Section 9.3),
2. Adds measurements into a 3-bit classical register,
3. Run it with `StatevectorSampler` at `shots=5000`,
4. Plot the resulting histogram.

Then, **without running any more code**, answer for yourself: what fraction of the 8 possible 3-bit outcomes should ever appear at all, and why? Add a short markdown or comment explanation in your own words.

In [ ]:
def build_capstone_circuit():
    # TODO: build the 3-qubit, 3-classical-bit GHZ-and-measure circuit described above
    raise NotImplementedError("Replace this with your circuit")

capstone_circuit = build_capstone_circuit()
sampler = StatevectorSampler()
job = sampler.run([capstone_circuit], shots=5000)
counts = job.result()[0].data.c.get_counts()
print(counts)
plot_histogram(counts, title="3-qubit GHZ state, 5000 shots")

# TODO: in a comment or markdown cell, explain in your own words why only two of the
# eight possible 3-bit outcomes ever appear, and roughly what fraction of shots each gets.

In [ ]:
# --- Check your answer ---
def check_capstone(qc):
    assert qc.num_qubits == 3 and qc.num_clbits == 3, "Need 3 qubits and 3 classical bits"
    sv_check = qc.copy()
    sv_check.remove_final_measurements()
    sv = Statevector(sv_check)
    probs = sv.probabilities_dict()
    assert set(probs.keys()) == {"000", "111"}, f"Expected only '000' and '111' to have nonzero amplitude, got {probs}"
    for outcome, p in probs.items():
        assert abs(p - 0.5) < 1e-6, f"Expected 50/50 split, got {probs}"
    print("Capstone passed! Only '000' and '111' appear, each ~50% of the time -- exactly the GHZ signature.")

check_capstone(capstone_circuit)

---
## 14. Solutions

Only look here after you've genuinely attempted each exercise above. Each solution is written as a standalone function; if you want to compare against your own answer, run the corresponding solution cell and then feed its output through the same `check_exercise_*` function used above.

In [ ]:
# --- Exercise 1 solution ---
def solution_exercise_1():
    qc = QuantumCircuit(1, 1)
    qc.x(0)
    qc.h(0)
    qc.measure(0, 0)
    return qc

sol1 = solution_exercise_1()
check_exercise_1(sol1)
sol1.draw(output="mpl")

In [ ]:
# --- Exercise 2 solution ---
def solution_exercise_2():
    qc = QuantumCircuit(1)
    qc.h(0)
    qc.z(0)
    qc.h(0)
    return qc

sol2 = solution_exercise_2()
check_exercise_2(sol2)
print(Statevector(sol2).data.round(3))
# Why this works: H|0> = |+>. Z|+> = |->  (Z flips the sign of the |1> component,
# which is what distinguishes |+> from |->). H|-> = |1>. So H-Z-H = X, exactly the
# identity mentioned in Section 4.4's exercise hint.

In [ ]:
# --- Exercise 3 solution ---
def solution_exercise_3():
    qc = QuantumCircuit(1, 1)
    qc.ry(np.pi/2, 0)
    qc.measure(0, 0)
    return qc

sol3 = solution_exercise_3()
check_exercise_3(sol3)

In [ ]:
# --- Exercise 4 solution ---
def solution_exercise_4():
    qc = QuantumCircuit(2)
    qc.x(1)        # flip qubit 1 first: turns the eventual |00>/|11> pair into |01>/|10>
    qc.h(0)
    qc.z(0)        # introduces the relative minus sign
    qc.cx(0, 1)
    return qc

sol4 = solution_exercise_4()
check_exercise_4(sol4)
print(Statevector(sol4).data.round(3))
# Walkthrough: start |00>. X on qubit1 -> |01> (using Qiskit's convention, |q1 q0> = |01>
# means q1=0,q0=1... double check against your own run above and Section 7's discussion
# if the bit you flipped doesn't match what you expected -- that's the ordering gotcha
# showing up in exactly the place it likes to bite people.
# H on qubit0 -> (|00> - |10>)/sqrt2 after the Z (Z|1>=-|1> supplies the minus sign)
# CX(0,1) then maps this into the |01>,|10> pair with the sign preserved.

In [ ]:
# --- Capstone solution ---
def solution_capstone():
    qc = QuantumCircuit(3, 3)
    qc.h(0)
    qc.cx(0, 1)
    qc.cx(1, 2)
    qc.measure([0, 1, 2], [0, 1, 2])
    return qc

sol_capstone = solution_capstone()
check_capstone(sol_capstone)

sampler = StatevectorSampler()
job = sampler.run([sol_capstone], shots=5000)
counts = job.result()[0].data.c.get_counts()
plot_histogram(counts, title="Capstone solution: 3-qubit GHZ")

# Explanation: only '000' and '111' have nonzero amplitude in a GHZ state, each with
# probability 1/2 -- so exactly 2 of the 8 possible 3-bit outcomes ever appear, each
# close to 2500/5000 shots, with the small deviation from an exact 50/50 split being
# ordinary statistical (shot) noise, per the discussion in Section 6.

---
## Where to go next

You now have working, tested intuition for: superposition and measurement, the standard single- and two-qubit gate set with their actual matrices, Qiskit's little-endian bit ordering, entanglement (and how to detect it two independent ways), and the current Sampler/Estimator primitives API. That covers the "basics" layer the v2.x certification assumes you have solid before it starts testing you on algorithms (VQE, QAOA, Grover, QFT-based algorithms) and hardware-aware topics (noise, error mitigation, pulse-level control).

A natural next notebook, if you want to keep going in this style, would be: parameterized circuits + `QAOAAnsatz`/`TwoLocal`, and a from-scratch VQE loop built directly on top of the `Estimator` primitive you just learned here.
